# Project 21 — Hybrid Retrieval Lab

## Compare BM25, Dense, and Hybrid Retrieval Locally

**Goal:** Implement three retrieval strategies — BM25 (sparse / keyword),
dense vector search, and hybrid (Reciprocal Rank Fusion) — then benchmark
them side-by-side on a labeled mini corpus to understand which query types
each method handles best.

**Stack:** Ollama · LangChain · ChromaDB · rank-bm25 · Jupyter

```
                        ┌──── BM25 (keyword) ────┐
User query ──►──────────┤                        ├──► Reciprocal Rank Fusion ──► top-k
                        └── Dense (embedding) ───┘
```

### What You'll Learn

1. Implement BM25 sparse retrieval from scratch
2. Build a dense vector retriever with ChromaDB + Ollama embeddings
3. Combine both with Reciprocal Rank Fusion (RRF)
4. Benchmark retrieval quality with ground-truth labels
5. Analyse which query types favour sparse vs dense vs hybrid
6. Run end-to-end QA comparison across all three strategies

### Prerequisites

- **Ollama** running locally with `nomic-embed-text` and `qwen3:8b`
- `pip install rank-bm25`
- Python 3.9+

In [ ]:
# Install dependencies (uncomment and run once)
!pip install -q langchain langchain-ollama langchain-community chromadb rank-bm25

## Step 1 — Verify Ollama Is Running

Before using any local model we confirm Ollama is reachable.

In [ ]:
import requests

OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"✅ Ollama is running — {len(models)} model(s) available")
except Exception as e:
    print(f"❌ Cannot reach Ollama at {OLLAMA_BASE}: {e}")

## Step 2 — Configure LLM, Embeddings, and Build Corpus

We create a **12-document** technical corpus covering different topics.
Each document has a unique `id` so we can measure retrieval accuracy
against ground-truth labels later.

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document

llm = ChatOllama(model="qwen3:8b", temperature=0.1)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

corpus = [
    Document(page_content="Python is a high-level programming language known for readability. "
        "It supports multiple paradigms including OOP and functional programming.",
        metadata={"id": 1, "topic": "python"}),
    Document(page_content="Rust provides memory safety without garbage collection through "
        "its ownership system. It competes with C++ for systems programming.",
        metadata={"id": 2, "topic": "rust"}),
    Document(page_content="Machine learning models learn patterns from data. Supervised "
        "learning uses labeled examples while unsupervised finds hidden structures.",
        metadata={"id": 3, "topic": "ml"}),
    Document(page_content="Vector databases store embeddings for similarity search. "
        "Popular options include ChromaDB, FAISS, Pinecone, and Weaviate.",
        metadata={"id": 4, "topic": "vectordb"}),
    Document(page_content="RAG combines retrieval with generation. The retriever finds "
        "relevant documents and the generator produces answers grounded in evidence.",
        metadata={"id": 5, "topic": "rag"}),
    Document(page_content="BM25 is a bag-of-words retrieval function that ranks documents "
        "based on term frequency and inverse document frequency with saturation.",
        metadata={"id": 6, "topic": "retrieval"}),
    Document(page_content="Transformers use self-attention to process sequences in parallel. "
        "The architecture consists of encoder and decoder stacks with multi-head attention.",
        metadata={"id": 7, "topic": "transformers"}),
    Document(page_content="Fine-tuning adapts a pre-trained model to specific tasks using "
        "smaller domain-specific datasets. LoRA reduces the parameters needed.",
        metadata={"id": 8, "topic": "finetuning"}),
    Document(page_content="Docker containers package applications with their dependencies. "
        "Images are built from Dockerfiles and run as isolated processes.",
        metadata={"id": 9, "topic": "docker"}),
    Document(page_content="Kubernetes orchestrates containerized workloads across clusters. "
        "Key concepts: pods, services, deployments, and horizontal pod autoscaling.",
        metadata={"id": 10, "topic": "kubernetes"}),
    Document(page_content="PostgreSQL is a relational database with ACID compliance. "
        "It supports JSON, full-text search, window functions, and CTEs.",
        metadata={"id": 11, "topic": "postgres"}),
    Document(page_content="Git is a distributed version control system. Branches, merges, "
        "rebases, and pull requests are core workflow concepts.",
        metadata={"id": 12, "topic": "git"}),
]
print(f"✅ Corpus: {len(corpus)} documents")

## Step 3 — BM25 Sparse Retriever

BM25 (Best Match 25) ranks documents by keyword overlap. It works well
when the user query contains exact terms that appear in the target
document — but struggles with paraphrases and synonyms.

In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np

tokenized_corpus = [doc.page_content.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query: str, k: int = 3):
    """Return top-k BM25 results with scores."""
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_k = np.argsort(scores)[::-1][:k]
    return [{"doc": corpus[idx], "score": float(scores[idx]), "rank": i + 1}
            for i, idx in enumerate(top_k)]

print("BM25 results for: 'vector database similarity search'")
for r in bm25_search("vector database similarity search"):
    print(f"  [{r['rank']}] score={r['score']:.3f} id={r['doc'].metadata['id']} "
          f"— {r['doc'].page_content[:60]}...")

## Step 4 — Dense Vector Retriever

Dense retrieval embeds both the query and documents into the same vector
space, then finds nearest neighbours. It handles paraphrases well but
may miss keyword-critical queries.

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(corpus, embeddings, collection_name="hybrid_lab")

def dense_search(query: str, k: int = 3):
    """Return top-k dense retrieval results with scores."""
    results = vectorstore.similarity_search_with_score(query, k=k)
    return [{"doc": doc, "score": float(score), "rank": i + 1}
            for i, (doc, score) in enumerate(results)]

print("Dense results for: 'vector database similarity search'")
for r in dense_search("vector database similarity search"):
    print(f"  [{r['rank']}] score={r['score']:.3f} id={r['doc'].metadata['id']} "
          f"— {r['doc'].page_content[:60]}...")

## Step 5 — Hybrid Retrieval with Reciprocal Rank Fusion

RRF combines ranked lists from multiple retrievers without needing
comparable scores. The formula is:

$$\text{RRF}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k$ is a smoothing constant (typically 60). Documents that appear
highly ranked in **both** lists get the highest combined score.

In [ ]:
def reciprocal_rank_fusion(result_lists, k=60):
    """Combine multiple ranked result lists using RRF."""
    fused_scores = {}
    labels = ["BM25", "Dense"]

    for list_idx, result_list in enumerate(result_lists):
        label = labels[list_idx] if list_idx < len(labels) else f"List{list_idx}"
        for r in result_list:
            doc_id = r["doc"].metadata["id"]
            rank = r["rank"]
            if doc_id not in fused_scores:
                fused_scores[doc_id] = {"doc": r["doc"], "score": 0.0, "sources": []}
            fused_scores[doc_id]["score"] += 1.0 / (k + rank)
            fused_scores[doc_id]["sources"].append(f"{label} rank={rank}")

    return sorted(fused_scores.values(), key=lambda x: x["score"], reverse=True)


def hybrid_search(query: str, k: int = 3):
    """BM25 + Dense → RRF hybrid search."""
    bm25_r = bm25_search(query, k=5)
    dense_r = dense_search(query, k=5)
    fused = reciprocal_rank_fusion([bm25_r, dense_r])
    return fused[:k]


print("Hybrid (RRF) results for: 'vector database similarity search'")
for i, r in enumerate(hybrid_search("vector database similarity search")):
    print(f"  [{i+1}] rrf={r['score']:.4f} id={r['doc'].metadata['id']} "
          f"— {r['doc'].page_content[:55]}...")
    print(f"       Sources: {r['sources']}")

## Step 6 — Benchmark with Ground-Truth Labels

We create test queries where we know the correct document(s). This lets
us compute **Hit@1** (does the correct doc appear in position 1?) and
**Hit@3** (does it appear in the top 3?) for each retrieval method.

In [ ]:
test_queries = [
    # (query, expected_doc_ids, query_type)
    ("What is BM25?", [6], "keyword"),
    ("How does RAG work?", [5], "keyword"),
    ("memory safe systems language", [2], "semantic"),
    ("neural network architecture with attention", [7], "semantic"),
    ("container orchestration platform", [10], "semantic"),
    ("relational database with JSON support", [11], "keyword+semantic"),
    ("version control branching", [12], "keyword"),
    ("adapting pre-trained models", [8], "semantic"),
]

def evaluate(search_fn, k=3):
    """Evaluate a search function on the test set."""
    hit_at_1 = 0
    hit_at_k = 0
    for query, expected, _ in test_queries:
        results = search_fn(query, k=k)
        result_ids = [r["doc"].metadata["id"] for r in results]
        if result_ids[0] in expected:
            hit_at_1 += 1
        if any(rid in expected for rid in result_ids):
            hit_at_k += 1
    n = len(test_queries)
    return {"hit@1": hit_at_1 / n, "hit@3": hit_at_k / n}

bm25_metrics = evaluate(bm25_search)
dense_metrics = evaluate(dense_search)
hybrid_metrics = evaluate(hybrid_search)

print(f"{'Method':<10} {'Hit@1':>8} {'Hit@3':>8}")
print("-" * 28)
print(f"{'BM25':<10} {bm25_metrics['hit@1']:>7.0%} {bm25_metrics['hit@3']:>7.0%}")
print(f"{'Dense':<10} {dense_metrics['hit@1']:>7.0%} {dense_metrics['hit@3']:>7.0%}")
print(f"{'Hybrid':<10} {hybrid_metrics['hit@1']:>7.0%} {hybrid_metrics['hit@3']:>7.0%}")

## Step 7 — Per-Query Breakdown

Let's see exactly where each method succeeds and fails. This reveals the
complementary strengths of sparse and dense retrieval.

In [ ]:
print(f"{'Query':<42} {'Type':<10} {'BM25':>5} {'Dense':>6} {'Hybrid':>7}")
print("-" * 75)

for query, expected, qtype in test_queries:
    bm25_top = bm25_search(query, k=1)[0]["doc"].metadata["id"]
    dense_top = dense_search(query, k=1)[0]["doc"].metadata["id"]
    hybrid_top = hybrid_search(query, k=1)[0]["doc"].metadata["id"]

    def mark(rid):
        return "✓" if rid in expected else "✗"

    print(f"{query[:40]:<42} {qtype:<10} {mark(bm25_top):>5} "
          f"{mark(dense_top):>6} {mark(hybrid_top):>7}")

## Step 8 — RRF Weight Sensitivity

RRF has one hyperparameter: the constant $k$ (default 60). Lower $k$
gives more weight to top-ranked results; higher $k$ smooths out the
ranking. Let's see how it affects accuracy.

In [ ]:
print(f"{'RRF k':<8} {'Hit@1':>8} {'Hit@3':>8}")
print("-" * 26)

for rrf_k in [10, 30, 60, 100, 200]:
    def hybrid_k(query, k=3, _rrf_k=rrf_k):
        bm25_r = bm25_search(query, k=5)
        dense_r = dense_search(query, k=5)
        fused = reciprocal_rank_fusion([bm25_r, dense_r], k=_rrf_k)
        return fused[:k]
    m = evaluate(hybrid_k)
    print(f"{rrf_k:<8} {m['hit@1']:>7.0%} {m['hit@3']:>7.0%}")

## Step 9 — End-to-End QA Comparison

Let's compare full RAG pipelines: retrieve with each method, then
generate an answer with the LLM. This shows how retrieval quality
propagates to answer quality.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY the provided context. Be concise."),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])
qa_chain = qa_prompt | llm | StrOutputParser()

def rag_pipeline(query, search_fn):
    """Retrieve → Generate answer."""
    results = search_fn(query, k=3)
    context = "\n".join([r["doc"].page_content for r in results])
    return qa_chain.invoke({"context": context, "question": query})

demo_queries = [
    "What is BM25 and how does it rank documents?",
    "How does Kubernetes handle scaling?",
    "What are the key concepts in version control?",
]

for q in demo_queries:
    print(f"\nQ: {q}")
    for name, fn in [("BM25", bm25_search), ("Dense", dense_search), ("Hybrid", hybrid_search)]:
        ans = rag_pipeline(q, fn)
        print(f"  [{name}] {str(ans)[:120]}...")

## Limitations and Tradeoffs

1. **Small corpus:** 12 documents is sufficient to illustrate patterns but
   real-world differences emerge more clearly at thousands of documents.

2. **BM25 tokenization:** We used simple whitespace splitting. Production
   BM25 should use stemming, stop-word removal, and better tokenization.

3. **Evaluation simplicity:** Hit@k is a basic metric. Production systems
   use nDCG, MAP, and MRR for finer-grained ranking evaluation.

4. **Latency not measured:** BM25 is near-instant; dense search adds
   embedding time; hybrid doubles retrieval cost.

5. **No learned fusion:** RRF is unsupervised. Learned-weight fusion
   (e.g., linear combination tuned on a dev set) can outperform RRF.

## How to Extend This Project

1. **Add a reranker:** Pipe hybrid results through a cross-encoder reranker
   (see Project 23) for a two-stage pipeline.

2. **Larger corpus:** Load a real dataset (e.g., Wikipedia passages) to see
   differences at scale.

3. **Better BM25:** Use Elasticsearch or `pyserini` for production-grade
   sparse retrieval with stemming and analyzers.

4. **Learned fusion weights:** Tune the BM25-vs-dense weight on a
   development set using Bayesian optimization.

5. **Latency dashboard:** Time each retrieval method and plot the
   accuracy-vs-latency Pareto frontier.

## What You Learned

| Concept | What We Did |
|---|---|
| **BM25 sparse retrieval** | Keyword-based ranking via term frequency |
| **Dense vector retrieval** | Semantic nearest-neighbour search |
| **Reciprocal Rank Fusion** | Unsupervised fusion of ranked lists |
| **Ground-truth benchmark** | Hit@1 / Hit@3 on labeled queries |
| **Per-query analysis** | Which query types favour which method |
| **Hyperparameter sensitivity** | Effect of RRF smoothing constant |
| **End-to-end QA comparison** | Retrieval quality → answer quality |